Варіант 6
Місто Чернігів

In [102]:
import pandas as pd

months = ["Січ","Лют","Бер","Кві","Тра","Чер","Лип","Сер","Вер","Жов","Лис","Гру"]

df = pd.DataFrame({
    "month": months,
    "avg_temp": [-5, -4, 1, 9, 15, 18, 20, 19, 13, 7, 1, -3],
    "precip": [40, 38, 40, 44, 56, 78, 86, 70, 50, 40, 48, 46],
})

print(df.head())
print(df.shape)
print(df.dtypes)

print("-"*6, "Завданння 1", "-"*6)

print(df.index)     # Початковий індекс
df.set_index("month", inplace=True)     # Зміна індекса на стовпець month
june = df.loc["Чер"]       # Повертає дані з Чер
print(june)
df.reset_index(inplace=True)    # Повертає звичайний індекс
print(df)

print("-"*6, "Завданння 2", "-"*6)

df.index=range(1, 13)
temp_first_half = df.loc[1:6, "avg_temp"]   # loc звертається до значення
print(temp_first_half)
iloc_temp_first_half = df.iloc[0:6, 1]     # iloc звертається до позиції
print(iloc_temp_first_half)
non_temp = df.loc[:, ["month", "precip"]]
print(non_temp)

print("-"*6, "Завданння 3", "-"*6)

warm = df.query("avg_temp > 15")
print(warm)
warm_rain = df.query("avg_temp > 10 and precip > 50")
print(warm_rain)
snowy = df.query("avg_temp < 0 or precip > 80")
print(snowy)

print("-"*6, "Завданння 4", "-"*6)

def season(month):
    if month in [12, 1, 2]:
        return "зима"
    elif month in [3, 4, 5]:
        return "весна"
    elif month in [6, 7, 8]:
        return "літо"
    else:
        return "осінь"

df["avg_temp_f"] = df["avg_temp"] * 9 / 5 + 32
df["is_wet"] = df["precip"] > df["precip"].mean()
df["season"] = df.index.map(season)
# ЯКЩО номер місяця записаний у окремому стовпці (наприклад, 'month_num'),
# то застосовуйте .apply() напряму до нього:
# df["season"] = df["month_num"].apply(get_season)
print(df)

  month  avg_temp  precip
0   Січ        -5      40
1   Лют        -4      38
2   Бер         1      40
3   Кві         9      44
4   Тра        15      56
(12, 3)
month         str
avg_temp    int64
precip      int64
dtype: object
------ Завданння 1 ------
RangeIndex(start=0, stop=12, step=1)
avg_temp    18
precip      78
Name: Чер, dtype: int64
   month  avg_temp  precip
0    Січ        -5      40
1    Лют        -4      38
2    Бер         1      40
3    Кві         9      44
4    Тра        15      56
5    Чер        18      78
6    Лип        20      86
7    Сер        19      70
8    Вер        13      50
9    Жов         7      40
10   Лис         1      48
11   Гру        -3      46
------ Завданння 2 ------
1    -5
2    -4
3     1
4     9
5    15
6    18
Name: avg_temp, dtype: int64
1    -5
2    -4
3     1
4     9
5    15
6    18
Name: avg_temp, dtype: int64
   month  precip
1    Січ      40
2    Лют      38
3    Бер      40
4    Кві      44
5    Тра      56
6    Чер      78
7

1. Чому .loc[1:6] та .iloc[0:6] повертають різну кількість рядків?
.loc працює за мітками (labels) і включає верхню межу. Наприклад, якщо індекс — це значення від 1 до 12, то .loc[1:6] шукає мітки 1, 2, 3, 4, 5, 6 (разом 6 рядків). Якщо ж індекс текстовий (наприклад, назви місяців Січ..Гру), то .loc[1:6] взагалі видасть помилку, бо чисел 1 і 6 немає серед міток.

.iloc працює за порядковими позиціями (0-indexed) і НЕ включает верхню межу (як стандартний slicing у Python). Запит .iloc[0:6] завжди бере елементи на позиціях 0, 1, 2, 3, 4, 5 (разом 6 рядків), незалежно від того, які індекси має таблиця.

Різниця в кількості виникає тоді, коли числові мітки індексу не збігаються з позиціями (наприклад, якщо індекс починається з 0, то .loc[1:6] поверне 6 рядків зі значеннями індексу 1..6, а .iloc[1:6] поверне 5 рядків на позиціях 1..5).

2. Чому and викликає помилку, а & — ні?
Ключове слово and (Python logical operator): очікує одне булеве значення (True або False). Оскільки вираз df["avg_temp"] > 10 повертає цілий об'єкт Pandas Series (масив булевих значень), Python не знає, як звести весь масив до одного булевого значення, і генерує ValueError: The truth value of a Series is ambiguous.

Оператор & (Bitwise AND operator): у Pandas перевантажений для виконання векторизованого поелементного порівняння ("Побітове І"). Він порівнює відповідні елементи двох Series між собою.

Дужки (...) & (...) обов'язкові через те, що оператор & має вищий пріоритет, ніж оператори порівняння >.

3. Різниця між обчислюваним стовпцем за формулою та через .apply()
Векторизована формула (наприклад, df["a"] + df["b"]):

Виконується на низькому рівні (C/NumPy).

Операції застосовуються одразу до всього масиву даних.

Працює максимально швидко і використовує менше пам'яті.

Створення через .apply(func):

Проходить по таблиці поінтервально або поінструкційно (фактично це прихований цикл for мовою Python).

Дозволяє використовувати складну власну логіку Python (if/else, обробку винятків, сторонні бібліотеки).

Працює значно повільніше на великих обсягах даних через накладні витрати інтерпретатора Python.